# OpenBind Blind Challenge 1, Zika NS2B-NS3 protease pose prediction

This notebook holds the whole pipeline in one place. Each section starts with a short note on what the next cell does.

**Goal.** Predict one protein-ligand complex for each of 400 compounds and submit 400 PDB files in one ZIP.
A pose succeeds only if BiSyRMSD is below 1.5 A, lDDT-PLI is above 0.8, and all PoseBusters checks pass.

**Deadlines.** Interim leaderboard 11 Nov 2026, final 16 Dec 2026 (both 23:59 UTC).

**Environments.** This notebook runs in `zika`. Tools that need a different stack are called through `run_in_env`.

| Env | Holds | Used for |
|---|---|---|
| `zika` | Boltz, PoseBusters, GNINA, Vina, RDKit, Meeko, torch | generating and checking poses (this kernel) |
| `zika-eval` | OpenStructure | the official scorer |
| `zika-md` | OpenMM, PDBFixer | pocket relaxation (CPU only) |

**Only one generated file.** The long Boltz batch must outlive this kernel, so section 6 writes a small bash script (`run_batch.sh`) and starts it in the background. Everything else is visible here.

## 1. Parameters
Change these values here. The cell is tagged `parameters` so papermill can override them.
`RUN` is the folder name under `runs/`. Use a new name for each experiment.
`LIMIT` builds only the first N compounds, which is handy for a quick test (for example `RUN = "test10"`, `LIMIT = 10`).
`ALLOW_LOW_MEMORY` lets the batch start when WSL has less than `MIN_WSL_GB`. Leave it `False` unless you are testing on purpose.

In [ ]:
RUN = "baseline"
LIMIT = None
SAMPLES = 5
CHUNK = 50
SEED = 42
MIN_WSL_GB = 9
ALLOW_LOW_MEMORY = False

## 2. Setup and paths
Imports, folder locations, and small helpers used everywhere.
`run_in_env` runs a command inside another conda env and returns the result.
`wsl_memory_gb` reads how much memory WSL actually has, which matters because a 6 GB cap killed the pilot run.

In [ ]:
import os, sys, re, json, shutil, subprocess, zipfile, datetime, inspect
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

ROOT = Path.home() / "OpenBindZika"
DATA = ROOT / "data"
RUNDIR = ROOT / "runs" / RUN
TUTORIAL = ROOT / "tutorial"
CSV = DATA / "challenge" / "openbind-blind-challenge-1-TEST-BLINDED.csv"
FASTA = TUTORIAL / "inputs" / "NS2B-NS3_protein_sequence.fasta"
MSA_NS2B = DATA / "msa" / "zika_ns2b.csv"
MSA_NS3 = DATA / "msa" / "zika_ns3.csv"
SCRATCH = ROOT / "runs" / "_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)
ENV_BIN = Path(sys.executable).parent


def fix_wsl_path(p):
    """Convert //wsl.localhost/Ubuntu/... paths from Windows dialogs to Linux paths."""
    p = str(p).replace("\\", "/")
    m = re.match(r"^//wsl(?:\.localhost|\$)/[^/]+(/.*)$", p)
    return Path(m.group(1)) if m else Path(p)


def wsl_memory_gb():
    """Total memory visible to WSL, in GB."""
    with open("/proc/meminfo") as f:
        kb = int(next(line for line in f if line.startswith("MemTotal")).split()[1])
    return kb / 1024 / 1024


def run_in_env(env, args, **kw):
    """Run a command inside another conda env, for example run_in_env('zika-eval', ['python', 'x.py'])."""
    return subprocess.run(["conda", "run", "-n", env, *args], capture_output=True, text=True, **kw)


print("run folder:", RUNDIR)
print("python:", sys.executable)

## 3. Environment health check
Confirms this kernel is the right env, that the GPU is visible, and how much memory WSL has.
Then `health_check` tests every tool in all three envs and prints a PASS or FAIL table.
This replaces `setup_check.py`. Run it after any install.

In [ ]:
import torch

print("env:", ENV_BIN.parent.name)
print("cuda available:", torch.cuda.is_available())
print(f"WSL memory: {wsl_memory_gb():.1f} GB (batch needs {MIN_WSL_GB}+)")
print("boltz:", shutil.which("boltz"), "| bust:", shutil.which("bust"), "| vina:", shutil.which("vina"))


def health_check():
    checks = [
        ("zika", "chem stack", 'python -c "import rdkit, meeko, Bio, pandas, sklearn"'),
        ("zika", "numpy 1.26", "python -c \"import numpy; assert numpy.__version__.startswith('1.26')\""),
        ("zika", "torch cuda", 'python -c "import torch; assert torch.cuda.is_available()"'),
        ("zika", "boltz cli", "boltz --help"),
        ("zika", "posebusters", "bust --help"),
        ("zika", "vina", "vina --version"),
        ("zika", "fpocket", "which fpocket"),
        ("zika", "gnina", "$HOME/gnina --version"),
        ("zika-eval", "openstructure", 'python -c "from ost.mol.alg.ligand_scoring_scrmsd import SCRMSDScorer; from ost.mol.alg.ligand_scoring_lddtpli import LDDTPLIScorer"'),
        ("zika-md", "openmm cpu", "python -c \"import openmm, pdbfixer; from openmm import Platform; Platform.getPlatformByName('CPU')\""),
    ]
    rows = []
    for env, name, cmd in checks:
        r = run_in_env(env, ["bash", "-c", cmd])
        rows.append({"env": env, "check": name, "result": "PASS" if r.returncode == 0 else "FAIL"})
    table = pd.DataFrame(rows)
    print(f"{(table.result == 'PASS').sum()}/{len(table)} passed")
    return table


# Uncomment to run (takes about a minute).
# health_check()

## 4. Challenge data
Loads the 400 compound SMILES and the NS2B and NS3 sequences from the tutorial FASTA.
The asserts catch mistakes early. NS2B must be 46 residues (chain A) and NS3 168 (chain B), matching the organizers' report.
The tutorial's own `blind_challenge_001.yaml` merged the two chains by mistake, so we always read the FASTA directly.

In [ ]:
def read_fasta(path):
    seqs = {}
    for rec in Path(path).read_text().split(">")[1:]:
        lines = rec.strip().split("\n")
        seqs[lines[0].split("|")[1]] = "".join(lines[1:])
    return seqs


seqs = read_fasta(FASTA)
NS2B, NS3 = seqs["Chain A"], seqs["Chain B"]
assert (len(NS2B), len(NS3)) == (46, 168), (len(NS2B), len(NS3))

df = pd.read_csv(CSV)
assert len(df) == 400 and df["Molecule_Name"].is_unique
mols = [Chem.MolFromSmiles(s) for s in df["SMILES"]]
bad = [n for n, m in zip(df["Molecule_Name"], mols) if m is None]
assert not bad, f"unparseable SMILES: {bad}"
df["heavy_atoms"] = [m.GetNumHeavyAtoms() for m in mols]

# The saved MSAs must start with the query sequence.
for msa, seq in [(MSA_NS2B, NS2B), (MSA_NS3, NS3)]:
    assert pd.read_csv(msa).loc[0, "sequence"] == seq, f"MSA does not match its sequence: {msa}"

print("NS2B", len(NS2B), "| NS3", len(NS3), "| compounds", len(df))
print(df["heavy_atoms"].describe().round(1).to_string())
df.head()

## 5. Boltz inputs
Builds one Boltz YAML per compound. Each has the two protein chains (A is NS2B, B is NS3) with the saved MSA files, plus the ligand as chain C.
Using saved MSAs means the ColabFold server is not called again.

In [ ]:
def boltz_yaml(smiles):
    return "\n".join([
        "version: 1",
        "sequences:",
        "  - protein:",
        "      id: A",
        f"      sequence: {NS2B}",
        f"      msa: {MSA_NS2B}",
        "  - protein:",
        "      id: B",
        f"      sequence: {NS3}",
        f"      msa: {MSA_NS3}",
        "  - ligand:",
        "      id: C",
        f"      smiles: {smiles}",
        "",
    ])


def build_inputs(frame, run, limit=None):
    out = ROOT / "runs" / run / "inputs"
    out.mkdir(parents=True, exist_ok=True)
    sub = frame.head(limit) if limit else frame
    for name, smi in zip(sub["Molecule_Name"], sub["SMILES"]):
        (out / f"{name}.yaml").write_text(boltz_yaml(smi))
    return len(sub)


n_inputs = build_inputs(df, RUN, LIMIT)
print(n_inputs, "inputs written to", RUNDIR / "inputs")

## 6. Boltz batch in chunks
Splits the inputs into chunks (folders of symlinks) and starts a background bash loop that runs `boltz predict` on one chunk at a time.
A chunk counts as finished when all its PDB files exist, and it then gets a `DONE` marker. A rerun skips finished chunks, so an interruption costs at most one chunk.
The loop judges success by counting files, not by exit code, because the pilot printed `Killed` after the results were already saved.
`launch_batch` refuses to start if WSL has too little memory or another batch is already running.

In [ ]:
def prepare_chunks(run, size):
    inputs = sorted((ROOT / "runs" / run / "inputs").glob("*.yaml"))
    for k in range(0, len(inputs), size):
        cdir = ROOT / "runs" / run / "chunks" / f"chunk_{k // size + 1:02d}"
        cdir.mkdir(parents=True, exist_ok=True)
        for f in inputs[k:k + size]:
            link = cdir / f.name
            if not link.exists():
                link.symlink_to(f)
    return len(list((ROOT / "runs" / run / "chunks").glob("chunk_*")))


def write_batch_script(run, samples, seed):
    rundir = ROOT / "runs" / run
    script = rundir / "run_batch.sh"
    script.write_text(f"""#!/usr/bin/env bash
cd {rundir}
for c in chunks/chunk_*; do
  n=$(basename "$c")
  out="out/$n"
  if [ -f "$out/DONE" ]; then echo "skip $n"; continue; fi
  boltz predict "$c" --out_dir "$out" --diffusion_samples {samples} --output_format pdb --seed {seed} --num_workers 1
  got=$(find "$out" -name '*_model_*.pdb' | wc -l)
  want=$(( $(ls "$c"/*.yaml | wc -l) * {samples} ))
  echo "$n pdbs=$got/$want"
  if [ "$got" -ge "$want" ]; then touch "$out/DONE"; else echo "incomplete, rerun to resume"; exit 1; fi
done
echo ALL CHUNKS DONE
""")
    return script


def launch_batch(run, samples, seed):
    if wsl_memory_gb() < MIN_WSL_GB and not ALLOW_LOW_MEMORY:
        raise RuntimeError(f"WSL has {wsl_memory_gb():.1f} GB. Restart WSL after setting memory=10GB in .wslconfig.")
    running = subprocess.run(["pgrep", "-f", "run_batch.sh"], capture_output=True, text=True).stdout.strip()
    if running:
        raise RuntimeError(f"A batch is already running (pid {running}).")
    script = write_batch_script(run, samples, seed)
    env = os.environ.copy()
    env["PATH"] = f"{ENV_BIN}:{env['PATH']}"
    log = open(ROOT / "runs" / run / "run.log", "a")
    p = subprocess.Popen(["bash", str(script)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=env)
    return p.pid


n_chunks = prepare_chunks(RUN, CHUNK)
print(n_chunks, "chunks prepared")

### Start the batch
This is the only cell that starts a long GPU job. Set Windows sleep to "never" first.
It runs in the background, so closing the notebook does not stop it. To resume after an interruption, run this cell again.

In [ ]:
# Uncomment to start.
# pid = launch_batch(RUN, SAMPLES, SEED)
# print("started, pid", pid)

### Progress
Shows how many chunks are finished, how many PDB files exist, and the last lines of the log. Run it whenever you want a status.

In [ ]:
def progress(run, samples=SAMPLES):
    base = ROOT / "runs" / run
    out = base / "out"
    n_chunks = len(list((base / "chunks").glob("chunk_*")))
    done = len(list(out.glob("chunk_*/DONE")))
    pdbs = len(list(out.rglob("*_model_*.pdb")))
    n_inputs = len(list((base / "inputs").glob("*.yaml")))
    print(f"chunks done {done}/{n_chunks} | PDB files {pdbs}/{n_inputs * samples}")
    log = base / "run.log"
    if log.exists():
        print("\n".join(log.read_text().splitlines()[-3:]))


progress(RUN)

## 7. PoseBusters check for every sample
For each Boltz sample this cell splits the complex into protein and ligand, rebuilds the ligand bond orders from the SMILES (a PDB file has none), and runs PoseBusters.
The result per sample is how many of the 22 checks passed, which ones failed, and Boltz's own confidence score.
`score_run` saves results to a CSV after every compound, so it resumes if interrupted.

In [ ]:
def find_pred_dir(run, compound):
    hits = list((ROOT / "runs" / run / "out").glob(f"*/boltz_results_*/predictions/{compound}"))
    return hits[0] if hits else None


def split_complex(pdb_path, tag):
    recs = [l for l in open(pdb_path) if l.startswith(("ATOM", "HETATM"))]
    lig = SCRATCH / f"{tag}_lig.pdb"
    prot = SCRATCH / f"{tag}_prot.pdb"
    lig.write_text("".join(l for l in recs if l[17:20].strip() == "LIG"))
    prot.write_text("".join(l for l in recs if l[17:20].strip() != "LIG"))
    return lig, prot


def ligand_sdf(smiles, lig_pdb, sdf_path):
    mol = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smiles), Chem.MolFromPDBFile(str(lig_pdb)))
    w = Chem.SDWriter(str(sdf_path))
    w.write(mol)
    w.close()


def posebusters(pdb_path, smiles, tag="pb"):
    """Return how many PoseBusters checks pass for one complex."""
    lig, prot = split_complex(pdb_path, tag)
    sdf = SCRATCH / f"{tag}_lig.sdf"
    ligand_sdf(smiles, lig, sdf)
    out_csv = SCRATCH / f"{tag}.csv"
    subprocess.run(["bust", str(sdf), "-p", str(prot), "--outfmt", "csv", "--output", str(out_csv)], capture_output=True, text=True)
    d = pd.read_csv(out_csv)
    checks = d.select_dtypes(include="bool")
    failed = [c for c in checks.columns if not checks[c].all()]
    return {"n_pass": int(checks.iloc[0].sum()), "n_total": int(checks.shape[1]), "failed": ";".join(failed)}


def score_compound(run, compound, smiles):
    pred = find_pred_dir(run, compound)
    if pred is None:
        return []
    rows = []
    for pdb in sorted(pred.glob(f"{compound}_model_*.pdb")):
        conf = json.loads((pred / f"confidence_{pdb.stem}.json").read_text())["confidence_score"]
        res = posebusters(pdb, smiles)
        rows.append({"compound": compound, "sample": pdb.stem.split("_")[-1], "confidence": round(conf, 4), "pdb": str(pdb), **res})
    return rows


def score_run(run, frame):
    """Run PoseBusters on every finished compound. Safe to rerun."""
    out_csv = ROOT / "runs" / run / "posebusters_samples.csv"
    have = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame()
    done = set(have["compound"]) if len(have) else set()
    for name, smi in tqdm(list(zip(frame["Molecule_Name"], frame["SMILES"]))):
        if name in done:
            continue
        rows = score_compound(run, name, smi)
        if rows:
            have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True)
            have.to_csv(out_csv, index=False)
    return have


# Uncomment to score all finished compounds (about 1 to 2 s per sample).
# samples_df = score_run(RUN, df)
# samples_df.groupby("compound").apply(lambda g: (g.n_pass == g.n_total).sum()).value_counts().sort_index()

## 8. Where does a failing pose clash
Lists the closest ligand to protein heavy-atom pairs in one PDB. A hydrogen-bonding N or O pair below about 2.3 A, or a carbon pair below about 3.0 A, is a likely clash.
Compound 001 fails `minimum_distance_to_protein` in all five samples, and this shows which atoms are involved.

In [ ]:
def read_atoms(pdb_path):
    rows = []
    for l in open(pdb_path):
        if l.startswith(("ATOM", "HETATM")):
            elem = l[76:78].strip() or l[12:16].strip()[0]
            if elem == "H":
                continue
            rows.append((l[12:16].strip(), l[17:20].strip(), l[22:26].strip(),
                         np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])))
    return rows


def closest_contacts(pdb_path, n=5):
    atoms = read_atoms(pdb_path)
    lig = [a for a in atoms if a[1] == "LIG"]
    prot = [a for a in atoms if a[1] != "LIG"]
    L = np.array([a[3] for a in lig])
    P = np.array([a[3] for a in prot])
    D = np.linalg.norm(L[:, None] - P[None], axis=2)
    idx = np.dstack(np.unravel_index(np.argsort(D, axis=None)[:n], D.shape))[0]
    return pd.DataFrame([{"ligand_atom": lig[i][0], "protein": f"{prot[j][1]}{prot[j][2]} {prot[j][0]}", "distance_A": round(float(D[i, j]), 2)} for i, j in idx])


# Example, uncomment after the pilot run exists:
# closest_contacts(next((ROOT / "runs/pilot/out").rglob("blind_challenge_001_model_0.pdb")))

## 9. Pick one pose per compound
The challenge needs exactly one pose per compound. The rule for now follows the organizers' baseline.
Keep samples that pass all checks, then take the one with the highest Boltz confidence.
If no sample passes, take the one with the most checks passed and flag it as not valid.
Later this cell is where the smarter selector (sample agreement, templates, relaxation) goes.

In [ ]:
def pick_best(samples):
    rows = []
    for comp, g in samples.groupby("compound"):
        g = g.assign(valid=g["n_pass"] == g["n_total"])
        g = g.sort_values(["valid", "n_pass", "confidence"], ascending=False)
        best = g.iloc[0]
        rows.append({"compound": comp, "sample": best["sample"], "valid": bool(best["valid"]),
                     "n_pass": int(best["n_pass"]), "failed": best["failed"], "confidence": best["confidence"],
                     "n_valid_samples": int(g["valid"].sum()), "pdb": best["pdb"]})
    return pd.DataFrame(rows)


# Quick self test of the rule with fake data.
_t = pd.DataFrame([
    {"compound": "a", "sample": "0", "confidence": 0.95, "n_pass": 21, "n_total": 22, "failed": "x", "pdb": "a0"},
    {"compound": "a", "sample": "1", "confidence": 0.94, "n_pass": 22, "n_total": 22, "failed": "", "pdb": "a1"},
    {"compound": "b", "sample": "0", "confidence": 0.90, "n_pass": 21, "n_total": 22, "failed": "x", "pdb": "b0"},
    {"compound": "b", "sample": "1", "confidence": 0.93, "n_pass": 21, "n_total": 22, "failed": "x", "pdb": "b1"},
])
_p = pick_best(_t).set_index("compound")
assert _p.loc["a", "pdb"] == "a1" and _p.loc["a", "valid"]
assert _p.loc["b", "pdb"] == "b1" and not _p.loc["b", "valid"]
print("selection rule OK")

# Uncomment after score_run has finished:
# selected = pick_best(samples_df)
# print(selected["valid"].value_counts())
# selected.to_csv(ROOT / "runs" / RUN / "selected.csv", index=False)

## 10. Official scorer and its self test
The organizers score with OpenStructure, which lives in `zika-eval`. This cell writes a tiny helper script to the scratch folder and runs it there.
The self test checks the scorer is trustworthy. A pose compared with itself must give BiSyRMSD near 0 and lDDT-PLI near 1, and a ligand shifted by 2 A must give BiSyRMSD of 2.
We only have reference structures for public complexes, so this is for local benchmarking, not for the challenge compounds.

In [ ]:
SCORER_PY = r'''
import sys, json
sys.path.insert(0, sys.argv[3])
from evaluation.evaluate_predictions import score_single_structure
res = score_single_structure(sys.argv[1], sys.argv[2])
print("RESULT " + json.dumps({k: float(v) for k, v in res.items()}))
'''


def score_vs_reference(model_pdb, ref_pdb):
    script = SCRATCH / "score_one.py"
    script.write_text(SCORER_PY)
    r = run_in_env("zika-eval", ["python", str(script), str(model_pdb), str(ref_pdb), str(TUTORIAL)], cwd=TUTORIAL)
    for line in r.stdout.splitlines():
        if line.startswith("RESULT "):
            return json.loads(line[7:])
    raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])


def scorer_self_test():
    ref = TUTORIAL / "outputs/my_structure_submission/blind_challenge_001.pdb"
    same = score_vs_reference(ref, ref)
    shifted = SCRATCH / "shifted.pdb"
    lines = []
    for l in ref.read_text().splitlines(True):
        if l.startswith(("ATOM", "HETATM")) and l[17:20].strip() == "LIG":
            l = l[:30] + f"{float(l[30:38]) + 2.0:8.3f}" + l[38:]
        lines.append(l)
    shifted.write_text("".join(lines))
    moved = score_vs_reference(shifted, ref)
    assert same["BiSyRMSD"] < 0.01 and same["LDDT-PLI"] > 0.99, same
    assert abs(moved["BiSyRMSD"] - 2.0) < 0.05, moved
    print("scorer OK | same:", same, "| shifted 2 A:", moved)


# Uncomment to run (takes under a minute).
# scorer_self_test()

## 11. Clash relaxation with OpenMM (to be added)
Compounds like 001 fail one PoseBusters check in every sample, so more samples will not fix them. The plan is to relax the protein pocket around the fixed ligand in OpenMM (env `zika-md`, CPU only) and rerun PoseBusters.
A relaxed pose only counts if the ligand moved less than 0.5 A and the backbone less than 0.3 A, and the poses that already pass must still pass.
The function goes here once the Claude Code session returns it. It will be called through `run_in_env("zika-md", ...)`.
Known issue. OpenMM CUDA fails on the laptop driver (PTX version error), so use the CPU platform.

## 12. Build the submission ZIP
Copies the chosen PDB for each compound into a `submission/` folder named `blind_challenge_NNN.pdb`, checks the basics, and zips them.
First look at how the organizers' example ZIP is laid out, so ours matches. Then read the official validator's signature before calling it, because I have not seen its arguments yet.

In [ ]:
example_zip = TUTORIAL / "outputs" / "example_structure_submission.zip"
names = zipfile.ZipFile(example_zip).namelist()
print(len(names), "files in the example zip")
print(names[:3])

sys.path.insert(0, str(TUTORIAL))
from validation.structure_validation import validate_structure_submission
print(inspect.signature(validate_structure_submission))
print(inspect.getdoc(validate_structure_submission))

### Build, check, and zip
`build_submission` copies the chosen PDBs. `check_submission` tests the file count, the names, and that each file has a `LIG` residue. `make_zip` writes a dated ZIP.
The final cell is commented out until `selected` covers all 400 compounds.

In [ ]:
def build_submission(selected, run):
    sub = ROOT / "runs" / run / "submission"
    sub.mkdir(parents=True, exist_ok=True)
    for _, r in selected.iterrows():
        shutil.copy(r["pdb"], sub / f"{r['compound']}.pdb")
    return sub


def check_submission(sub, frame):
    files = sorted(Path(sub).glob("blind_challenge_*.pdb"))
    problems = []
    if len(files) != 400:
        problems.append(f"{len(files)} files, expected 400")
    missing = set(frame["Molecule_Name"]) - {f.stem for f in files}
    if missing:
        problems.append(f"missing: {sorted(missing)[:5]} ...")
    for f in files:
        resn = {l[17:20].strip() for l in f.read_text().splitlines() if l.startswith(("ATOM", "HETATM"))}
        if "LIG" not in resn:
            problems.append(f"{f.name} has no LIG residue")
    return problems


def make_zip(sub, run):
    name = ROOT / "runs" / run / f"submission_{datetime.date.today():%Y%m%d}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(Path(sub).glob("*.pdb")):
            z.write(f, f.name)
    return name


# Uncomment when selected.csv covers all 400 compounds.
# sub = build_submission(selected, RUN)
# print(check_submission(sub, df) or "basic checks passed")
# zip_path = make_zip(sub, RUN)
# print(zip_path)
# Then run the official validator on the zip, using the signature printed above, before uploading.

## 13. Decisions and findings so far
A running record, which also feeds the methods report.

- Boltz-2 on the RTX 4070 laptop takes about 43 s per compound for 5 samples (3-compound pilot).
- WSL was capped at 6 GB and a 3-compound run was killed after its results were written. The cap is now 10 GB in `.wslconfig` and applies after `wsl --shutdown`.
- PoseBusters on the pilot. Compound 002 had 4 of 5 valid samples, 003 had 2 of 5, and 001 had 0 of 5.
- All five 001 samples fail `minimum_distance_to_protein`. The tutorial's own example pose for 001 fails the same check (ligand N to the Tyr115 backbone O at 2.18 A).
- The organizers' Boltz baseline reached 88.25 percent of compounds with a full 22-check pass by picking among 5 samples.
- fpocket ranked the true binding site third on a Boltz model, so the site is defined from template ligands, not pocket rank.
- The scorer self test passed. Identical pose gave BiSyRMSD near 0, and a 2 A shift gave 2.0 with lDDT-PLI 0.69.
- Package pins in `zika`. Boltz needs `click==8.1.7` and numpy 1.26.4. OpenMM and OpenStructure live in their own envs because they want numpy 2.x.
- The tutorial YAML for compound 001 merged NS2B and NS3 into one chain. Our builder reads the FASTA instead.